# Lab: Catch the made-up answer
**Solution notebook.** Reference solution. Try the starter notebook first.

## What you will do
You will check an AI answer against the source document it was meant to use, and flag sentences with details the source does not support. This is a simple first version of the grounding checks you will build for RAG in Module 6.

**Time:** 20 minutes. **Needs:** Google Colab or Python 3.12+. No API key and no extra libraries.

The clinic, the policy and the answer are fictional and were written for this course. The topic is billing administration only.

In [ ]:
# Step 1. The source document and the AI answer to check.
SOURCE = """Riverside Family Clinic: Billing and Payment Policy (fictional)
Patients receive a statement 14 days after their visit.
Payment is due within 30 days of the statement date.
The clinic accepts card, bank transfer and payment plans for balances over $200.
Payment plans can run for up to 6 months with no interest.
Questions go to the billing office at extension 4410, Monday to Friday."""

AI_ANSWER = """You will receive a statement 14 days after your visit.
Payment is due within 45 days of the statement date.
Payment plans are available for balances over $200 and can run for up to 12 months.
A late fee of $25 applies after the due date.
You can call the billing office at extension 4410 on weekdays."""

In [ ]:
# Step 2. Pull out the checkable details in a sentence: numbers and amounts.
import re

def details(sentence: str) -> set:
    """Return the set of numbers (with a $ sign if present) found in the sentence."""
    return set(re.findall(r"\$?\d+", sentence))

print(sorted(details("Payment plans for balances over $200 can run for 6 months")))

In [ ]:
# Step 3. Word overlap: what share of a sentence's longer words appear in the source?
def word_support(sentence: str, source: str) -> float:
    """Share (0 to 1) of the sentence's words of 5+ letters that also appear in the source."""
    src_words = set(re.findall(r"[a-z]+", source.lower()))
    words = [w for w in re.findall(r"[a-z]+", sentence.lower()) if len(w) >= 5]
    if not words:
        return 1.0
    return round(sum(w in src_words for w in words) / len(words), 2)

In [ ]:
# Step 4. Check every sentence in the answer.
def check_answer(answer: str, source: str, min_support: float = 0.6):
    source_details = details(source)
    flagged = 0
    for line in [s.strip() for s in answer.splitlines() if s.strip()]:
        unsupported = sorted(details(line) - source_details)
        support = word_support(line, source)
        problems = []
        if unsupported:
            problems.append("details not in source: " + ", ".join(unsupported))
        if support < min_support:
            problems.append(f"low word support ({support})")
        flagged += bool(problems)
        print(("CHECK " if problems else "ok    ") + line)
        for p in problems:
            print("      -> " + p)
    print(f"\n{flagged} of {len([s for s in answer.splitlines() if s.strip()])} sentences need a human check")

check_answer(AI_ANSWER, SOURCE)

## Check your understanding
1. Which made-up details did the checker catch? Which kind of made-up claim could it miss?
2. The answer sounded confident in every sentence. What does that tell you about using tone to judge accuracy?

## Try to beat the checker
Add a sentence to `AI_ANSWER` that is false but contains no new numbers, for example a made-up rule about deposits. Run Step 4 again. Did word support catch it? Write down what a better check would need.

## Apply it to your field
Paste a short, non-confidential policy from your track into `SOURCE`. Ask an AI tool a question about it, paste the answer into `AI_ANSWER`, and run the check. Record any flagged sentence and whether it was wrong.

---
BJS Academy · Prompt to Agent (Course 03) · Dr. Bih Janet Shufor. Original course material. Learn AI. Shape tomorrow.